# Encyclopedia header — `derivatives_test` (07_nbody_rebound_rust)

This is the **SolveIt_Notebooks_for_rust encyclopedia copy** of this notebook. Everything below is exact and copy-paste runnable from the **workspace root** (the folder that contains both `rustSolveIt_macos-silicon_SUNDIALS_7_8_0/` and `SolveIt_Notebooks_for_rust/`).

## 1. Execute this notebook

One-time build of the pure-Rust engine behind it:
```bash
cd rustSolveIt_macos-silicon_SUNDIALS_7_8_0/reboundx_rust && cargo build --release
```
Batch execution of this copy (writes real outputs back into it):
```bash
python3 SolveIt_Notebooks_for_rust/_tools/run_copy.py SolveIt_Notebooks_for_rust/07_nbody_rebound_rust/derivatives_test/derivatives_test.ipynb
```
Interactive execution (Shift+Enter through every cell, Python 3 (ipykernel) kernel):
```bash
python3 -m pip install --user jupyterlab   # once
cd SolveIt_Notebooks_for_rust/07_nbody_rebound_rust/derivatives_test && jupyter lab derivatives_test.ipynb
```

## 2. Display the browser GUI

This family's artifacts are the executed notebook and the files it writes (see section 4); where a browser GUI exists the companion player opens it:
```bash
python3 SolveIt_Notebooks_for_rust/07_nbody_rebound_rust/derivatives_test/player_derivatives_test.py gui
```

## 3. Movies

No pre-recorded movie pairs with this notebook. To watch it as a movie: open the live scene window (section 2) — the scene plays in real time — and screen-record it (macOS: Cmd+Shift+5). The 13 `video_*` notebooks in `05_mechanism_videos/` are the family with recorded movie pages.

## 4. Database / data access

This notebook's data is its **executed transcript** (every command sent to the pure-Rust engine and every reply, embedded as real cell outputs) — there is no SQLite database in this family. Export the transcript (and any paired artifacts) with:
```bash
python3 SolveIt_Notebooks_for_rust/07_nbody_rebound_rust/derivatives_test/player_derivatives_test.py data exported_data
```

## 5. The full player (companion script, shipped in this folder)

`player_derivatives_test.py` sits next to this notebook. It re-executes the notebook (`run`), opens the browser GUI (`gui`), **jumps to the key event** (`jump`), captures a PNG of the GUI headlessly (`capture out.png`, needs Google Chrome), and exports the data (`data outdir`). It is standard-library Python and never computes physics — it drives and displays the recorded, verified artifacts. The complete script, verbatim:

```python
#!/usr/bin/env python3
"""player_derivatives_test.py — full player for derivatives_test.ipynb (standard library only).

Modes
  python3 player_derivatives_test.py run              re-execute this notebook copy
  python3 player_derivatives_test.py gui              open the browser GUI
  python3 player_derivatives_test.py jump             open the GUI jumped to the key
                                           event (the N-body run's final state)
  python3 player_derivatives_test.py capture out.png  save a PNG of the GUI
                                           (headless Chrome)
  python3 player_derivatives_test.py data outdir      export this notebook's data

Family: rebound.  Everything runs against recorded, verified artifacts of
the pure-Rust SUNDIALS engine; the player never computes physics itself.
"""

import http.server
import json
import os
import shutil
import socket
import subprocess
import sys
import tempfile
import threading
import time
import webbrowser
from pathlib import Path

HERE = Path(__file__).resolve().parent
WS = HERE.parents[2]                       # workspace root
ENGINE = WS / "rustSolveIt_macos-silicon_SUNDIALS_7_8_0"
if not ENGINE.exists():
    ENGINE = WS                # fresh clone: the repository root IS the engine
TAG = "derivatives_test"
FAMILY = "rebound"
GUI_PAGE = None                      # main browser-GUI artifact (or None)
MOVIE = None                            # recorded movie page (or None)
GUI_SERVER = None                  # live-GUI server.py (or None)
POSIM_SCRIPT = None              # paired .posim scene (or None)
DBS = []                                # SQLite database files
CAPTURE_BUTTON = None          # id of the page's jump button


def chrome():
    for c in ("/Applications/Google Chrome.app/Contents/MacOS/Google Chrome",
              shutil.which("google-chrome") or "",
              shutil.which("chromium") or ""):
        if c and Path(c).exists():
            return c
    raise SystemExit("headless capture needs Google Chrome installed")


def free_port():
    s = socket.socket()
    s.bind(("127.0.0.1", 0))
    p = s.getsockname()[1]
    s.close()
    return p


def serve(directory, page_name, jump=False):
    """Serve `directory` on an ephemeral port; return the page URL.
    With jump=True, serve an augmented copy that clicks the page's own
    jump-to-capture button after load (the original file is untouched)."""
    import functools
    port = free_port()
    workdir = directory
    name = page_name
    if jump and CAPTURE_BUTTON:
        tmp = Path(tempfile.mkdtemp(prefix="player_" + TAG + "_"))
        body = (Path(directory) / page_name).read_text(encoding="utf-8")
        body += ('<script>window.addEventListener("load",function(){'
                 'var b=document.getElementById("' + CAPTURE_BUTTON + '");'
                 'if(b){b.click();}});</script>')
        (tmp / name).write_text(body, encoding="utf-8")
        workdir = tmp
    handler = functools.partial(http.server.SimpleHTTPRequestHandler,
                                directory=str(workdir))
    httpd = http.server.ThreadingHTTPServer(("127.0.0.1", port), handler)
    threading.Thread(target=httpd.serve_forever, daemon=True).start()
    return f"http://127.0.0.1:{port}/{name}", httpd


def scene_replay_url():
    """Replay the paired .posim scene into a fresh posim child and open its
    live scene window server; returns (url, process)."""
    binary = os.environ.get("POSIM_BIN", str(ENGINE / "target" / "release" / "posim"))
    if not Path(binary).is_file():
        raise SystemExit("build the engine first: cd rustSolveIt_macos-silicon_"
                         "SUNDIALS_7_8_0 && cargo build --release -p posim")
    port = free_port()
    proc = subprocess.Popen([binary, "--machine"],
                            stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                            text=True, bufsize=1,
                            env=dict(os.environ, POSIM_NO_BROWSER="1"))
    def rpc(cmd):
        proc.stdin.write(json.dumps({"op": "exec", "code": cmd}) + "\n")
        proc.stdin.flush()
        while True:
            line = proc.stdout.readline()
            if not line:
                raise RuntimeError("posim closed the connection")
            r = json.loads(line)
            if "event" in r:          # asynchronous scene notice, not a reply
                continue
            if not r.get("ok"):
                raise RuntimeError("posim refused %r: %s" % (cmd, r.get("error")))
            return r
    # Commands are line-based, but DEF blocks span lines: accumulate until
    # braces balance and send each complete statement as one code string.
    pending = []
    depth = 0
    for raw in Path(POSIM_SCRIPT).read_text(encoding="utf-8").splitlines():
        stripped = raw.strip()
        if not pending and (not stripped or stripped.startswith("#")):
            continue
        pending.append(raw)
        depth += raw.count("{") - raw.count("}")
        if depth <= 0:
            rpc("\n".join(pending))
            pending = []
            depth = 0
    if pending:
        rpc("\n".join(pending))
    rpc(f"scene create {port}")
    time.sleep(0.5)
    return f"http://127.0.0.1:{port}/", proc


def open_gui(jump=False):
    if FAMILY in ("solveit", "dynamic", "collision") or (
            FAMILY == "video" and GUI_PAGE is None and MOVIE is None):
        if not POSIM_SCRIPT:
            print("this notebook has no paired .posim scene; its verified")
            print("artifact is the executed notebook itself — open it with:")
            print("  jupyter lab", TAG + ".ipynb")
            return
        url, proc = scene_replay_url()
        print("live posim scene window:", url)
        print("(the paired scene", Path(POSIM_SCRIPT).name,
              "was replayed; Ctrl-C to quit)")
        webbrowser.open(url)
        try:
            proc.wait()
        except KeyboardInterrupt:
            proc.terminate()
        return
    if GUI_SERVER and not jump:
        print("starting the live GUI server (Ctrl-C to quit):", GUI_SERVER)
        subprocess.run([sys.executable, str(GUI_SERVER)], cwd=str(ENGINE))
        return
    page = GUI_PAGE or MOVIE
    if page is None:
        print("this notebook family has no browser GUI; its verified artifact")
        print("is the executed notebook itself — open it with: jupyter lab",
              TAG + ".ipynb")
        return
    url, httpd = serve(str(Path(page).parent), Path(page).name, jump=jump)
    print(("jumped to " + "the N-body run's final state" + ": ") if jump else "GUI: ", url)
    webbrowser.open(url)
    print("serving; Ctrl-C to quit")
    try:
        while True:
            time.sleep(3600)
    except KeyboardInterrupt:
        httpd.shutdown()


def capture(out_png, jump=True):
    c = chrome()
    if FAMILY in ("solveit", "dynamic", "collision") or (
            FAMILY == "video" and GUI_PAGE is None and MOVIE is None):
        if not POSIM_SCRIPT:
            raise SystemExit("no paired .posim scene recorded for this notebook")
        url, proc = scene_replay_url()
        try:
            subprocess.run([c, "--headless=new", "--disable-gpu",
                            "--window-size=1500,950",
                            "--screenshot=" + str(out_png), url],
                           check=True, capture_output=True, timeout=120)
        finally:
            proc.terminate()
    else:
        page = GUI_PAGE or MOVIE
        if page is None:
            raise SystemExit("no browser GUI to capture for this notebook")
        url, httpd = serve(str(Path(page).parent), Path(page).name, jump=jump)
        time.sleep(0.3)
        subprocess.run([c, "--headless=new", "--disable-gpu",
                        "--window-size=1500,950", "--virtual-time-budget=8000",
                        "--screenshot=" + str(out_png), url],
                       check=True, capture_output=True, timeout=120)
        httpd.shutdown()
    print("captured:", out_png)


def export_data(outdir):
    out = Path(outdir)
    out.mkdir(parents=True, exist_ok=True)
    for db in DBS:
        import csv
        import sqlite3
        con = sqlite3.connect(db)
        cur = con.cursor()
        tables = [r[0] for r in cur.execute(
            "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
        for t in tables:
            cur.execute(f"SELECT * FROM {t}")
            cols = [d[0] for d in cur.description]
            rows = cur.fetchall()
            with open(out / f"{Path(db).stem}_{t}.csv", "w", newline="",
                      encoding="utf-8") as f:
                w = csv.writer(f)
                w.writerow(cols)
                w.writerows(rows)
        con.close()
        print("exported", len(tables), "tables from", Path(db).name)
    nb = json.loads((HERE / (TAG + ".ipynb")).read_text(encoding="utf-8"))
    with open(out / "transcript.txt", "w", encoding="utf-8") as f:
        for cell in nb["cells"]:
            if cell["cell_type"] != "code":
                continue
            f.write("### In:\n" + "".join(cell["source"]) + "\n### Out:\n")
            for o in cell.get("outputs", []):
                f.write("".join(o.get("text", [])))
            f.write("\n" + "=" * 70 + "\n")
    print("exported executed transcript ->", out / "transcript.txt")
    if MOVIE:
        shutil.copyfile(MOVIE, out / Path(MOVIE).name)
        print("copied movie page ->", out / Path(MOVIE).name)


def main():
    mode = sys.argv[1] if len(sys.argv) > 1 else "gui"
    if mode == "run":
        sys.exit(subprocess.run([sys.executable,
                                 str(WS / "SolveIt_Notebooks_for_rust" / "_tools" / "run_copy.py"),
                                 str(HERE / (TAG + ".ipynb"))]).returncode)
    elif mode == "gui":
        open_gui(jump=False)
    elif mode == "jump":
        open_gui(jump=True)
    elif mode == "capture":
        capture(sys.argv[2] if len(sys.argv) > 2 else TAG + ".png")
    elif mode == "data":
        export_data(sys.argv[2] if len(sys.argv) > 2 else "exported_data")
    else:
        print(__doc__)
        sys.exit(2)


if __name__ == "__main__":
    main()
```

# derivatives_test — 65 orbital derivative functions

Evaluates all 65 reb_particle_derivative_* functions on two configurations and dumps raw bits; verified 130/130 lines bit-identical against the C build.

This notebook is self-contained: it builds the example with cargo, runs it, and shows the result. Everything it does can also be done by hand in a terminal:

```
cd rebound_rust
cargo build --release --example derivatives_test
cd porttest
../target/release/examples/derivatives_test
```

In [1]:
import os, subprocess
EXE = ".exe" if os.name == "nt" else ""   # platform executable suffix
NB_DIR  = os.getcwd()                       # <crate>/notebooks
ROOT    = os.path.dirname(os.path.dirname(NB_DIR))
CRATE   = os.path.join(ROOT, "rebound_rust")
WORK    = os.path.join(CRATE, "porttest")
if not os.path.exists(os.path.join(CRATE, "Cargo.toml")):
    raise SystemExit(
        "Could not find the crate. Run this notebook from "
        "the notebooks folder of a full checkout: " + CRATE)
# The example that is BUILT and RUN. It is usually the one the
# notebook is named after; where it differs (the stock
# shearing_sheet integrates forever by design) the terminating
# variant is used instead, and the note above says so.
EXAMPLE = "derivatives_test"
OUTFILE = None
os.makedirs(WORK, exist_ok=True)
res = subprocess.run(["cargo", "build", "--release", "--example", EXAMPLE],
                     cwd=CRATE, capture_output=True, text=True)
print(res.stderr.strip()[-400:] or "build ok")

   Compiling rebound_rs v5.1.1 (/Users/nsh/Developer/github/rustSolveIt_macos-silicon_SUNDIALS_7_8_0/rustSolveIt_macos-silicon_SUNDIALS_7_8_0/rebound_rust)
    Finished `release` profile [optimized] target(s) in 0.11s


In [2]:
exe = os.path.join(CRATE, "target", "release", "examples", EXAMPLE + EXE)
res = subprocess.run([exe], cwd=WORK, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print("STDERR:", res.stderr[-2000:])

derivatives_rust.txt written (65 functions x 2 configs)



In [3]:
p = os.path.join(WORK, OUTFILE) if OUTFILE else None
if p and os.path.exists(p):
    lines = open(p).read().splitlines()
    print(f"{len(lines)} result lines; first 6:")
    print("\n".join(lines[:6]))
